In [11]:
from dlfs.base import Module, Loss, Optimizer, Layer

from dlfs.modules import SequentialWrapper
from dlfs.layers import DenseLayer, ConvolutionalLayer
from dlfs.activation import ReLU, Sigmoid

from dlfs.loss import MSE_Loss, BCE_Loss
from dlfs.optimizers import Optimizer_Adam
from dlfs.helpers import dilate, pad_to_shape

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

In [12]:
def im2col_multi(X, kernel_shape, stride=1, padding=(0, 0)):
    B = X.shape[0]
    kH, kW = kernel_shape

    if isinstance(padding, tuple):
        pad_H, pad_W = padding
    else:
        pad_H = pad_W = padding

    X_padded = np.pad(X, ( (0, 0), (0, 0),(pad_H, pad_H), (pad_W, pad_W) ), mode='constant')

    H_p, W_p = X_padded.shape[2:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    cols = []

    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W * stride, stride):
                patch = X_padded[b, :, i:i+kH, j:j+kW].ravel()
                cols.append(patch)
    
    return np.array(cols), out_H, out_W

def col2im_multi(cols, output_shape, kernel_shape, stride=1, padding=0):
    B, C, H, W = output_shape
    kH, kW = kernel_shape
    H_p, W_p = H+2*padding, W+2*padding
    X_padded = np.zeros((B, C, H_p, W_p))

    out_H = (H_p - kH)//stride + 1
    out_W = (W_p - kW)//stride + 1

    idx = 0
    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W*stride, stride):
                patch = cols[idx].reshape(C, kH, kW)
                X_padded[b, :, i:i+kH, j:j+kW] += patch
                idx += 1

    if padding>0:
        X_padded = X_padded[:, :, padding:-padding, padding:-padding]

    return X_padded

def conv2d(X, W, stride=1, padding=0):
    
    C_out, C_in, kH, kW = W.shape
    B = X.shape[0]
    X_col, out_H, out_W = im2col_multi(X, (kH, kW), stride, padding)

    W_col = W.reshape(C_out, -1)
    Y_col = X_col @ W_col.T

    Y = Y_col.T.reshape(B, C_out, out_H, out_W)
    return Y

def conv_transpose2d(Y, W, stride=1, padding=0, output_shape=None):
    C_out, C_in, kH, kW = W.shape
    B = Y.shape[0]
    Y_col = Y.reshape(C_out, -1)
    W_col = W.reshape(C_out, -1)
    X_col = W_col.T @ Y_col

    if output_shape is None:
        H_out = (Y.shape[2]-1) * stride - 2*padding + kH
        W_out = (Y.shape[3]-1) * stride - 2*padding + kW
        output_shape = (B, C_in, H_out, W_out)

    X = col2im_multi(X_col.T, output_shape=output_shape, kernel_shape=(kH, kW), stride=stride, padding=padding)

    return X

def conv_transpose_backward(delta):
    #delta_col = im2col_multi(delta, )
    pass

In [13]:
class ConvTransposeLayer(Layer):

    def __init__(self, input_channels: tuple, output_channels: int, kernel_size: int, stride: int = 1, padding: int = 0, output_padding=0) -> None:

        self.input_channels = input_channels
        self.output_channels = output_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.output_padding = output_padding

        # Create output and kernel shapes
        self.kernel_size = kernel_size

        # Initialize layer parameters
        self.kernels = np.random.randn(input_channels, output_channels, kernel_size, kernel_size)
        self.biases = np.random.randn(output_channels)

    def forward(self, inputs: np.ndarray, training=False) -> None:
        C_in, C_out, kH, kW = self.kernels.shape
        B = inputs.shape[0]
        self.Y_col = inputs.reshape(C_in, -1)
        self.K_col = self.kernels.reshape(C_in, -1)
        X_col = self.K_col.T @ self.Y_col

        H_out = (inputs.shape[2]-1) * self.stride - 2*self.padding + kH
        W_out = (inputs.shape[3]-1) * self.stride - 2*self.padding + kW
        output_shape = (B, C_out, H_out, W_out)
        print(f'output shape forward {output_shape}')

        X = col2im_multi(X_col.T, output_shape=output_shape, kernel_shape=(kH, kW), stride=self.stride, padding=self.padding)
        self.output = X


    def backward(self, delta: np.ndarray) -> None:
        C_out, C_in, kH, kW = self.kernels.shape
        delta_col, _, _ = im2col_multi(delta, kernel_shape=(kH, kW), stride=self.stride, padding=self.padding)

        print(f'delta_col: {delta_col.shape}, y_col: {self.Y_col.shape}, kernel_col: {self.K_col.shape}')

        self.dkernels = self.Y_col @ delta_col
        self.dinputs = self.K_col @ delta_col.T

        self.dkernels = self.dkernels.reshape(C_out, C_in, kH, kW)

    def get_parameters(self):
        param_names = ["kernels", "biases"]
        return super()._filter_parameters(param_names)

In [14]:
x = np.random.randn(5, 3, 28, 28)

layer1 = ConvolutionalLayer(input_shape=(3, 28, 28), output_channels=6, kernel_size=3, stride=1, padding=0)
layer2 = ConvTransposeLayer(input_channels = 6, output_channels = 3, kernel_size=3, stride=1, padding=0, output_padding=(4, 5))

layer1.forward(x)

print(layer1.output.shape)

layer2.forward(layer1.output)

print(layer2.output.shape)

(5, 6, 26, 26)
output shape forward (5, 3, 28, 28)
(5, 3, 28, 28)


In [15]:
delta = np.random.randn(*layer2.output.shape)

layer2.backward(delta)

delta_col: (3380, 27), y_col: (6, 3380), kernel_col: (6, 27)


In [16]:
print(f'grads: {layer2.dinputs.shape} | {layer2.dkernels.shape}')

grads: (6, 3380) | (6, 3, 3, 3)
